Imports 

In [1]:
import pandas as pd 
from pathlib import Path
import re
from collections import Counter

import unicodedata

from difflib import SequenceMatcher


Olá caros leitores,  esse notebook é motivado pelos achados das duas ultimas explorações de dados: 

Avaliando a divisão da estrutura semanticas, descobrimos que, algumas obras que não são cartas, nem discursos e não haviam sido pegas na primeira análise de dados. 

Descobrimos poesias e cânticos, evidencias no notebook: 
[AED/ExploreEstrategiaExtracaoCapitulos.ipynb](AED/ExploreEstrategiaExtracaoCapitulos.ipynb)

E acho que essas obras merecem uma classificação a parte. Para não nos complicarmos demais irei nesse notebook reavaliar e distribuir as obras de Machado em algumas categorias

* Romances (contos/livros)
* Cartas
* Criticas e discursos
* Poemas


Novamente nossa dificuldade maior é que não temos esses dados anotados, e na nossa primeira análise de dados, verificamos que não temos essa info direta no portal de dominío público, portanto seguiremos atráves de heuristicas. 


Esse notebook irá atualizar o pipeline da bronze, já que estamos retrabalhando a etapa de classificação de obras, dado nossos novos achados, e assim é a vida em dados

ser cético de modo saudavel com os dados

bora lá

Carregando dados: 

In [2]:
PROJECT_ROOT = Path("..")

METADATA_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "metadata"
    / "bronze"
    / "obras_machado_assis_bronze.csv"
)


df_obras = pd.read_csv(
    METADATA_PATH
)

print(
    "Total no metadata:",
    len(df_obras),
)

df_obras

Total no metadata: 257


,titulo,autor,fonte,acessos,pdf_path,bronze_text_path,qtd_assinaturas_machado,is_carta,qtd_titulos_similares,titulos_similares,is_coletanea
0,A Carteira,Machado de Assis,[ua] Universidade da Amazônia - UNAMA,114102,Dataset/raw/A_Carteira.pdf,Dataset/bronze/A_Carteira.txt,0,False,0,[],False
1,A Cartomante,Machado de Assis,[ua] Universidade da Amazônia - UNAMA,154851,Dataset/raw/A_Cartomante.pdf,Dataset/bronze/A_Cartomante.txt,0,False,0,[],False
2,A Causa Secreta,Machado de Assis,[ua] Universidade da Amazônia - UNAMA,53202,Dataset/raw/A_Causa_Secreta.pdf,Dataset/bronze/A_Causa_Secreta.txt,0,False,0,[],False
3,A chave,Machado de Assis,[fs] Universidade Federal de Santa Catarina - ...,55648,Dataset/raw/A_chave.pdf,Dataset/bronze/A_chave.txt,0,False,0,[],False
4,A Chinela Turca,Machado de Assis,[bv] Biblioteca Virtual do Estudante Brasileir...,63578,Dataset/raw/A_Chinela_Turca.pdf,Dataset/bronze/A_Chinela_Turca.txt,0,False,1,"[""Só""]",False
...,...,...,...,...,...,...,...,...,...,...,...
252,Viver!,Machado de Assis,[bv] Biblioteca Virtual do Estudante Brasileir...,11632,Dataset/raw/Viver!.pdf,Dataset/bronze/Viver!.txt,0,False,0,[],False
253,Várias Histórias,Machado de Assis,[bv] Biblioteca Virtual do Estudante Brasileir...,24726,Dataset/raw/Várias_Histórias.pdf,Dataset/bronze/Várias_Histórias.txt,2,False,17,"[""A Cartomante"", ""A Causa Secreta"", ""A Desejad...",True
254,Vênus! divina vênus!,Machado de Assis,[fs] Universidade Federal de Santa Catarina - ...,8184,Dataset/raw/Vênus!_divina_vênus!.pdf,Dataset/bronze/Vênus!_divina_vênus!.txt,0,False,0,[],False
255,Última receita,Machado de Assis,[fs] Universidade Federal de Santa Catarina - ...,5506,Dataset/raw/Última_receita.pdf,Dataset/bronze/Última_receita.txt,0,False,0,[],False


In [3]:
qtd_cartas = df_obras["is_carta"].sum()
qtd_coletaneas = df_obras["is_coletanea"].sum()

print(
    f"Total de obras: {len(df_obras)}"
)
print(
    f"Cartas: {qtd_cartas}"
)
print(
    f"Coletâneas: {qtd_coletaneas}"
)

Total de obras: 257
Cartas: 2
Coletâneas: 9


Até então 2 cartas e 9 coletâneas

Vamos as famigeradas heuristicas

Sabemos dado nossas análise anterior que : 

São critícas: 

* Textos críticos
* Críticas Teatrais

São poemas/cânticos : 

* Americanas
* Falenas
* O almada



Ambos tipos tem seu cápitulos marcados via algarismos romanos acho que é um indicativo forte pra basearmos a heuristica

##### Heuristicas Poemas 

Avaliando poemas e poesias, e até canticos, eles tem algo em comum que aprendemos na escola: 

* A estrutura textual em forma de versos e estrofes


Isso faz com os versos tenham tamanhos similares entre si, além de serem curtos 



Podemos transformar isso em homogenidade no tamanho das linhas, linhas que terminam em \n e um tamanho máximo de palavras por linha 

Vamos antes olhar a obra Americanas, falenas e O almada brevemente na primeira página 

In [4]:
def mostrar_inicio_bronze(
    df_obras: pd.DataFrame,
    titulos: list[str],
    qtd_linhas: int = 40,
):
    for titulo in titulos:

        resultado = df_obras[
            df_obras["titulo"]
            .str.casefold()
            == titulo.casefold()
        ]

        if resultado.empty:
            print(
                f"[!] Obra não encontrada: {titulo}"
            )
            continue

        row = resultado.iloc[0]

        txt_path = (
            PROJECT_ROOT
            / row["bronze_text_path"]
        )

        linhas = txt_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).splitlines()

        print("=" * 100)
        print(titulo)
        print("=" * 100)

        print(
            "\n".join(
                linhas[:qtd_linhas]
            )
        )

        print()

In [5]:
mostrar_inicio_bronze(
    df_obras,
    [
        "Americanas",
        "Falenas",
        "O Almada",
    ],
)

Americanas
...filha melhor do Eterno, América!
G. Dias, Timb., c. III.
POTIRA
***
...Os Tamoios, entre outras presas que fizeram, levaram esta índia, a qual
pretendeu o capitão da empresa violar: resistiu valorosamente dizendo em língua
brasílica: “Eu sou cristã e casada; não hei de fazer traição a Deus e a meu marido;
bem podes matar-me e fazer de mim o que quiserdes.” Deu-se por afrontado o
bárbaro, e em vingança lhe acabou a vida com grande crueldade.
Vasc. Chr. da Companhia de Jesus, liv 3º

2
POTIRA1

Se, poi ch’a morte il corpo le percosse,
Desse almen vita alla memoria d’ella.
ARIOSTO, Orl. Fur., c. XXIX, est. XXXI

I

Moça cristã das solidões antigas,
Em que áurea folha reviveu teu nome?
Nem o eco das matas seculares,
Nem a voz das sonoras cachoeiras,
O transmitiu aos séculos futuros.
Assim da tarde estiva às auras frouxas
Tênue fumo do colmo no ar se perde;
Nem de outra sorte em moribundos lábios
A humana voz expira. O horror e o sangue
Da miseranda cena em que, de envolta
Co’

Excluindo a primeira página que são algumas anotações e avisos de Assis, antes dos poemas proprieamente ditos, encontramos a estrutura esperada 

Vamos avaliar o tamanho desses carinhas pra ter uma ideia base de quais parametros setar na nossa heuristica que encontra poemas e cânticos

In [6]:
def analisar_linhas_poeticas(
    df_obras: pd.DataFrame,
    titulos: list[str],
) -> pd.DataFrame:

    registros = []

    for titulo in titulos:

        resultado = df_obras[
            df_obras["titulo"]
            .str.casefold()
            == titulo.casefold()
        ]

        if resultado.empty:
            print(
                f"[!] Obra não encontrada: {titulo}"
            )
            continue

        row = resultado.iloc[0]

        txt_path = (
            PROJECT_ROOT
            / row["bronze_text_path"]
        )

        if not txt_path.exists():
            print(
                f"[!] Bronze não encontrado: {txt_path}"
            )
            continue

        texto = txt_path.read_text(
            encoding="utf-8",
            errors="ignore",
        )

        for indice_linha, linha_original in enumerate(
            texto.splitlines()
        ):

            linha = linha_original.strip()

            if not linha:
                continue

            registros.append({
                "obra": titulo,
                "indice_linha": indice_linha,
                "linha": linha,
                "qtd_chars": len(linha),
                "qtd_palavras": len(
                    linha.split()
                ),
            })

    return pd.DataFrame(
        registros
    )

In [7]:
df_linhas_poeticas = (
    analisar_linhas_poeticas(
        df_obras,
        [
            "Americanas",
            "Falenas",
            "O Almada",
        ],
    )
)
#agrupando as stats por obra
df_linhas_poeticas.groupby(
    "obra"
)[
    [
        "qtd_chars",
        "qtd_palavras",
    ]
].agg([
    "mean",
    "median",
    "std",
    "min",
    "max",
]).round(1)

qtd_chars                       qtd_palavras                    
                mean median   std min  max         mean median  std min max
obra                                                                       
Americanas      34.1   34.0  15.8   1  118          6.3    6.0  2.8   1  22
Falenas         30.5   34.0  14.0   1  111          5.6    6.0  2.8   1  37
O Almada        36.4   34.0  16.0   1   93          6.5    6.0  3.0   1  21

A três obras estão semalhantes entre si, o que indica que de fato poemas e canticos possuem uma estrutura similar em quantidade de palavras e vamos conseguir usar isso a nosso favor, vamos avaliar o agrupamento geral das 3: 

In [8]:
df_linhas_poeticas[
    [
        "qtd_chars",
        "qtd_palavras",
    ]
].agg([
    "mean",
    "median",
    "std",
    "min",
    "max",
]).round(1)

,qtd_chars,qtd_palavras
mean,33.5,6.1
median,34.0,6.0
std,15.4,2.9
min,1.0,1.0
max,118.0,37.0


Seguimos com valores proximos ao individual, e temos uns edges cases de 37 palavras em uma linhas, mas isso dever ser algum aviso ou outro pedaçõ do texto não pertencente ao poema em si, vamos avaliar os percentis: 

In [9]:
df_linhas_poeticas[
    [
        "qtd_chars",
        "qtd_palavras",
    ]
].quantile(
    [
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
    ]
)

,qtd_chars,qtd_palavras
0.50,34.0,6.0
0.75,38.0,7.0
0.90,42.0,8.0
0.95,49.0,10.0
0.99,97.0,17.0


Supimpa! Até 0.99 das linhas desses caras tem menos que 17 palavras, e até 0.95 temos menos que 10 palavras, então além de 37 palavras ser um outlier, vamos conseguir classificar a obra com essa ideia e heuristica 

In [10]:
def avaliar_estrutura_poetica(
    txt_path,
    max_palavras_linha: int = 10,
    min_pct_linhas_curtas: float = 0.85,
) -> dict:

    txt_path = Path(txt_path)

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    linhas = [
        linha.strip()
        for linha in texto.splitlines()
        if linha.strip()
    ]

    if not linhas:
        return {
            "is_poema": False,
            "qtd_linhas": 0,
            "qtd_linhas_curtas": 0,
            "pct_linhas_curtas": 0,
        }

    qtd_linhas_curtas = sum(
        len(linha.split())
        <= max_palavras_linha
        for linha in linhas
    )

    pct_linhas_curtas = (
        qtd_linhas_curtas
        / len(linhas)
    )

    return {
        "is_poema": (
            pct_linhas_curtas
            >= min_pct_linhas_curtas
        ),
        "qtd_linhas": len(linhas),
        "qtd_linhas_curtas": qtd_linhas_curtas,
        "pct_linhas_curtas": pct_linhas_curtas,
    }

Heuristica que avalia que até 85% das linhas da obra tem que ser menores que 10 chars 

In [11]:
for titulo in [
    "Americanas",
    "Falenas",
    "O Almada",
]:

    row = df_obras[
        df_obras["titulo"].str.casefold()
        == titulo.casefold()
    ].iloc[0]

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    resultado = avaliar_estrutura_poetica(
        txt_path
    )

    print(
        f"{titulo}: "
        f"{resultado['pct_linhas_curtas']:.2%} "
        f"| poema={resultado['is_poema']}"
    )

Americanas: 96.06% | poema=True
Falenas: 97.68% | poema=True
O Almada: 91.90% | poema=True


Pegou corretamente nossos 3 casos 

Quero antes fazer uma análise bem permissiva e ver o que vai pegar, vamos testar com 51% de linhas menores que 10 palavras

In [12]:
resultados_poemas_051 = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        continue

    resultado = avaliar_estrutura_poetica(
        txt_path,
        min_pct_linhas_curtas=0.51
    )

    resultados_poemas_051.append({
        "titulo": row["titulo"],
        **resultado,
    })


df_poemas_051 = pd.DataFrame(
    resultados_poemas_051
)
df_poemas_051[
    df_poemas_051["is_poema"]
].sort_values(
    "pct_linhas_curtas",
    ascending=False,
)

,titulo,is_poema,qtd_linhas,qtd_linhas_curtas,pct_linhas_curtas
174,Ocidentais,True,1454,1445,0.993810
186,Poesias dispersas,True,1465,1454,0.992491
81,Falenas,True,3273,3197,0.976780
138,O Bote de rapé,True,219,213,0.972603
30,Antes da Missa,True,242,235,0.971074
238,Uma ode de anacreonte,True,817,792,0.969400
178,Os deuses de casaca,True,1296,1248,0.962963
27,Americanas,True,2916,2801,0.960562
133,O almada,True,2679,2462,0.919000
58,Crisálidas,True,2427,2172,0.894932


Pegamos várias coisas que não são poeams, como Linha Reta e Linha Curva, que caiu na malha fina por ser uma obra curta com vários dialogos. 

Além disso pegamos um outro tipo de obra curto, similar a estrutura dos poeamas: peças teatrais

Como : 
*  Forças Caudinas
*  Hoje Avental, amanhã Luva
*  O protocolo 


Felizmentes este tem outra cara fácil de reconhecermos que é as linhas começarem com uma mesma palavra que indica o locutor(a) da vez na fala e isso irá nos ajudar a classificar estas . 

Avaliando manualmente, o limite do que ainda é poema ou  cântico são mais que 85% de linhas menores que 10 palavras, menos que isso e começamos a esbarrar em outras coisas. Fecharei esse escopo aqui sob risco de aumentar demais as heuristicas e complicar as coisas hahaha . 

A ideia é termos um bom nível de classificação, mas não classificar 100% corretamente a coletanea de machado de assis, ceús, nem expertise técnica linguística tenho pra isso !



In [13]:
resultados_poemas = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        continue

    resultado = avaliar_estrutura_poetica(
        txt_path,
        min_pct_linhas_curtas=0.85
    )

    resultados_poemas.append({
        "titulo": row["titulo"],
        **resultado,
    })


df_poemas = pd.DataFrame(
    resultados_poemas
)
df_poemas[
    df_poemas["is_poema"]
].sort_values(
    "pct_linhas_curtas",
    ascending=False,
)

,titulo,is_poema,qtd_linhas,qtd_linhas_curtas,pct_linhas_curtas
174,Ocidentais,True,1454,1445,0.993810
186,Poesias dispersas,True,1465,1454,0.992491
81,Falenas,True,3273,3197,0.976780
138,O Bote de rapé,True,219,213,0.972603
30,Antes da Missa,True,242,235,0.971074
238,Uma ode de anacreonte,True,817,792,0.969400
178,Os deuses de casaca,True,1296,1248,0.962963
27,Americanas,True,2916,2801,0.960562
133,O almada,True,2679,2462,0.919000
58,Crisálidas,True,2427,2172,0.894932


Fechamos aqui, bora pra próxima 

##### Teatro 

O texto teatral tem uma característica marcante: o nome do personagem aparece antes da fala. 

e.g 

LULU: Está um belo dia

ou algo assim 

Maria: 

Que dia belo de fato! 


Avaliando a repetição dessa marca conseguimos identificar textos com essas caracterisitcas teatrais

In [14]:
PALAVRAS_ESTRUTURA_TEATRO = {
    "cena",
    "ato",
    "quadro",
    "prólogo",
    "prologo",
    "epílogo",
    "epilogo",
}


def normalizar_locutor(
    linha: str,
) -> str:

    linha = linha.strip(
        ".,;:!?\"'()[]{}—–- "
    )

    return re.sub(
        r"\s+",
        " ",
        linha.casefold(),
    )


def linha_parece_locutor(
    linha: str,
    max_palavras: int = 4,
    max_chars: int = 40,
) -> bool:

    linha = linha.strip()

    if not linha:
        return False

    if len(linha) > max_chars:
        return False

    palavras = linha.split()

    if len(palavras) > max_palavras:
        return False

    primeira_palavra = (
        palavras[0]
        .strip(".,;:!?()[]")
        .casefold()
    )

    if (
        primeira_palavra
        in PALAVRAS_ESTRUTURA_TEATRO
    ):
        return False

    letras = [
        char
        for char in linha
        if char.isalpha()
    ]

    if not letras:
        return False

    proporcao_maiusculas = (
        sum(
            char.isupper()
            for char in letras
        )
        / len(letras)
    )

    return (
        proporcao_maiusculas
        >= 0.90
    )


def extrair_locutor_linha(
    linha: str,
) -> str | None:

    linha = linha.strip()

    if not linha:
        return None

    # ==========================================
    # CASO 1
    # Locutor sozinho na linha.
    #
    # LULU
    # VENÂNCIO
    # BOTA DIREITA
    # ==========================================

    if linha_parece_locutor(
        linha
    ):
        return normalizar_locutor(
            linha
        )

    # ==========================================
    # CASO 2
    # Locutor e fala na mesma linha.
    #
    # LULU — Eu sou assim...
    # VENÂNCIO: Ah! e a senhora?
    # ELISA - Eu sei lá!
    # ==========================================

    match = re.match(
        r"""
        ^
        (?P<locutor>
            [A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ]
            [A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ\s.'-]{0,39}?
        )
        \s*
        (?:
            :|
            —|
            –|
            -
        )
        \s*
        (?P<fala>\S.+)
        $
        """,
        linha,
        flags=re.VERBOSE,
    )

    if not match:
        return None

    locutor = (
        match.group(
            "locutor"
        )
        .strip()
    )

    if not linha_parece_locutor(
        locutor
    ):
        return None

    return normalizar_locutor(
        locutor
    )


In [15]:
def avaliar_estrutura_teatral(
    txt_path,
    min_repeticoes_locutor: int = 3,
    min_locutores: int = 2,
    min_turnos: int = 10,
    min_pct_locutores_recorrentes: float = 0.60,
) -> dict:

    txt_path = Path(txt_path)

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    linhas = [
        linha.strip()
        for linha in texto.splitlines()
        if linha.strip()
    ]

    candidatos = []

    for indice, linha in enumerate(
        linhas
    ):

        locutor = extrair_locutor_linha(
            linha
        )

        if locutor is None:
            continue

        candidatos.append({
            "indice": indice,
            "linha": linha,
            "locutor": locutor,
        })

    contagem = Counter(
        candidato["locutor"]
        for candidato in candidatos
    )

    locutores_recorrentes = {
        locutor
        for locutor, qtd
        in contagem.items()
        if qtd >= min_repeticoes_locutor
    }

    marcadores_recorrentes = [
        candidato
        for candidato in candidatos
        if (
            candidato["locutor"]
            in locutores_recorrentes
        )
    ]

    pct_locutores_recorrentes = (
        len(marcadores_recorrentes)
        / len(candidatos)
        if candidatos
        else 0
    )

    qtd_turnos = len(
        marcadores_recorrentes
    )

    is_teatro = (
        len(locutores_recorrentes)
        >= min_locutores
        and qtd_turnos
        >= min_turnos
        and pct_locutores_recorrentes
        >= min_pct_locutores_recorrentes
    )

    return {
        "is_teatro": is_teatro,
        "qtd_candidatos_locutor": len(
            candidatos
        ),
        "qtd_locutores_recorrentes": len(
            locutores_recorrentes
        ),
        "qtd_turnos": qtd_turnos,
        "pct_locutores_recorrentes": (
            pct_locutores_recorrentes
        ),
        "locutores": sorted(
            locutores_recorrentes
        ),
        "contagem_locutores": {
            locutor: contagem[locutor]
            for locutor
            in sorted(
                locutores_recorrentes
            )
        },
    }

Avaliando nas 3 obras que sabemos se tratar de autos teatrais

In [16]:
titulos_teatro = [
    "As Forças Caudinas",
    "Hoje Avental, amanhã Luva",
    "O protocolo",
]

for titulo in titulos_teatro:

    resultado = df_obras[
        df_obras["titulo"]
        .str.casefold()
        == titulo.casefold()
    ]

    if resultado.empty:
        print(
            f"[!] Não encontrada: {titulo}"
        )
        continue

    row = resultado.iloc[0]

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    analise = avaliar_estrutura_teatral(
        txt_path
    )

    print("=" * 80)
    print(titulo)

    print(
        "Teatro:",
        analise["is_teatro"],
    )

    print(
        "Locutores recorrentes:",
        analise[
            "qtd_locutores_recorrentes"
        ],
    )

    print(
        "Turnos:",
        analise["qtd_turnos"],
    )

    print(
        "Pct. candidatos recorrentes:",
        f"{analise['pct_locutores_recorrentes']:.2%}",
    )

    print(
        "Locutores:",
        analise["contagem_locutores"],
    )

    print()

As Forças Caudinas
Teatro: True
Locutores recorrentes: 5
Turnos: 652
Pct. candidatos recorrentes: 97.31%
Locutores: {'coronel': 64, 'emília': 180, 'margarida': 121, 'seabra': 79, 'tito': 208}

Hoje Avental, amanhã Luva
Teatro: True
Locutores recorrentes: 3
Turnos: 290
Pct. candidatos recorrentes: 97.97%
Locutores: {'bento': 38, 'durval': 121, 'rosinha': 131}

O protocolo
Teatro: True
Locutores recorrentes: 4
Turnos: 436
Pct. candidatos recorrentes: 95.61%
Locutores: {'elisa': 126, 'lulu': 96, 'pinheiro': 118, 'venâncio': 96}



Bom com um heuristica que avalia locutores recorrentes, conseguimos classificar bem as 3 que tinhamos previamente, todas tinham mais que 90% de candidatos recorrentes vamos testar com essa e ver se encontramos mais peças no conjunto todo 


Mais uma vez vamos começar permissivos 

In [17]:
resultados_teatro = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        continue

    analise = avaliar_estrutura_teatral(
        txt_path
    )

    resultados_teatro.append({
        "titulo": row["titulo"],
        **analise,
    })


df_teatro = pd.DataFrame(
    resultados_teatro
)

df_teatro[
    df_teatro["is_teatro"]
].sort_values(
    "pct_locutores_recorrentes",
    ascending=False,
)

,titulo,is_teatro,qtd_candidatos_locutor,qtd_locutores_recorrentes,qtd_turnos,pct_locutores_recorrentes,locutores,contagem_locutores
108,Lição de Botânica,True,405,4,403,0.995062,"[barão, d. cecília, d. helena, d. leonor]","{'barão': 117, 'd. cecília': 60, 'd. helena': ..."
100,"Hoje Avental, amanhã Luva",True,296,3,290,0.979730,"[bento, durval, rosinha]","{'bento': 38, 'durval': 121, 'rosinha': 131}"
134,O Anel de Polícrates,True,46,2,45,0.978261,"[a, z]","{'a': 23, 'z': 22}"
66,Desencantos,True,611,4,595,0.973813,"[clara, clara, pedro alves, luís, pedro alves]","{'clara': 204, 'clara, pedro alves': 3, 'luís'..."
35,As Forças Caudinas,True,670,5,652,0.973134,"[coronel, emília, margarida, seabra, tito]","{'coronel': 64, 'emília': 180, 'margarida': 12..."
140,O caminho da porta,True,402,4,390,0.970149,"[carlota, doutor, inocêncio, valentim]","{'carlota': 122, 'doutor': 104, 'inocêncio': 4..."
178,Os deuses de casaca,True,314,7,302,0.961783,"[apolo, cupido, júpiter, marte, mercúrio, prot...","{'apolo': 40, 'cupido': 64, 'júpiter': 70, 'ma..."
165,O Protocolo,True,456,4,436,0.956140,"[elisa, lulu, pinheiro, venâncio]","{'elisa': 126, 'lulu': 96, 'pinheiro': 118, 'v..."
195,Quase ministro,True,316,8,301,0.952532,"[agapito, bastos, martins, mateus, müller, pac...","{'agapito': 16, 'bastos': 29, 'martins': 82, '..."
238,Uma ode de anacreonte,True,221,4,209,0.945701,"[cléon, lisias, lísias, mirto]","{'cléon': 76, 'lisias': 4, 'lísias': 60, 'mirt..."


Ora vejam só, pegaram várias coisas que se repetem , mas não são nomes de personagens na obra 

* como notas semanais, que pegou os capitulos
* Correspondencia de machado de assis, que epgou suas assinaturas

mas tirando isso o qeu tem mais que 91% são de fato peças escritas
e aqui temos um caso interessante na coletanea Novas Reliquias (descobrimos essa nas primeiras análises) que pegou bota direita e esquerda como personagens e de fato o são, machado escrevia contos e peças teatrais as vezes como outros objetos ou até animais, então não tá errado!


Vou consertar a heuristica pra pegar mais que 91% agora e evitar capitulo e normalizar os nomes também 

In [18]:
PALAVRAS_ESTRUTURA_TEATRO = {
    "cena",
    "ato",
    "quadro",
    "prólogo",
    "prologo",
    "epílogo",
    "epilogo",
    "capítulo",
    "capitulo",
    "canto",
}


def normalizar_locutor(
    linha: str,
) -> str:

    linha = linha.strip().casefold()

    # Remove pontuação nas extremidades.
    linha = linha.strip(
        ".,;:!?\"'()[]{}—–-"
    )

    # Normaliza espaços.
    linha = re.sub(
        r"\s+",
        " ",
        linha,
    )

    return linha


def linha_parece_locutor(
    linha: str,
    max_palavras: int = 4,
    max_chars: int = 40,
) -> bool:

    linha = linha.strip()

    if not linha:
        return False

    if len(linha) > max_chars:
        return False

    palavras = linha.split()

    if len(palavras) > max_palavras:
        return False

    primeira_palavra = (
        palavras[0]
        .strip(
            ".,;:!?\"'()[]{}—–-"
        )
        .casefold()
    )

    # Evita confundir estruturas como:
    #
    # CAPÍTULO I
    # CANTO II
    # CENA III
    # ATO I
    if (
        primeira_palavra
        in PALAVRAS_ESTRUTURA_TEATRO
    ):
        return False

    letras = [
        char
        for char in linha
        if char.isalpha()
    ]

    if not letras:
        return False

    proporcao_maiusculas = (
        sum(
            char.isupper()
            for char in letras
        )
        / len(letras)
    )

    return (
        proporcao_maiusculas
        >= 0.90
    )


def extrair_locutor_linha(
    linha: str,
) -> str | None:

    linha = linha.strip()

    if not linha:
        return None

    # ==========================================
    # CASO 1
    # Locutor ocupa a linha inteira.
    #
    # LULU
    # VENÂNCIO
    # BOTA DIREITA
    # ==========================================

    if linha_parece_locutor(
        linha
    ):
        return normalizar_locutor(
            linha
        )

    # ==========================================
    # CASO 2
    # Locutor e fala estão na mesma linha.
    #
    # LULU — Eu sou assim...
    # VENÂNCIO: Ah! e a senhora?
    # ELISA - Eu sei lá!
    # ==========================================

    match = re.match(
        r"""
        ^
        (?P<locutor>
            [A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ]
            [A-ZÁÀÂÃÉÊÍÓÔÕÚÜÇ\s.'-]{0,39}?
        )
        \s*
        (?:
            :|
            \.|
            —|
            –|
            -
        )
        \s*
        (?P<fala>\S.+)
        $
        """,
        linha,
        flags=re.VERBOSE,
    )

    if not match:
        return None

    locutor = (
        match.group(
            "locutor"
        )
        .strip()
    )

    # Aplica as mesmas regras ao nome
    # encontrado no início da linha.
    if not linha_parece_locutor(
        locutor
    ):
        return None

    return normalizar_locutor(
        locutor
    )


def avaliar_estrutura_teatral(
    txt_path,
    min_repeticoes_locutor: int = 3,
    min_locutores: int = 2,
    min_turnos: int = 10,
    min_pct_locutores_recorrentes: float = 0.91,
) -> dict:

    txt_path = Path(txt_path)

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    linhas = [
        linha.strip()
        for linha in texto.splitlines()
        if linha.strip()
    ]

    candidatos = []

    for indice, linha in enumerate(
        linhas
    ):

        locutor = extrair_locutor_linha(
            linha
        )

        if locutor is None:
            continue

        candidatos.append({
            "indice": indice,
            "linha": linha,
            "locutor": locutor,
        })

    contagem = Counter(
        candidato["locutor"]
        for candidato in candidatos
    )

    locutores_recorrentes = {
        locutor
        for locutor, qtd
        in contagem.items()
        if qtd >= min_repeticoes_locutor
    }

    turnos_recorrentes = [
        candidato
        for candidato in candidatos
        if (
            candidato["locutor"]
            in locutores_recorrentes
        )
    ]

    qtd_turnos = len(
        turnos_recorrentes
    )

    pct_locutores_recorrentes = (
        qtd_turnos
        / len(candidatos)
        if candidatos
        else 0
    )

    is_teatro = (
        len(locutores_recorrentes)
        >= min_locutores
        and qtd_turnos
        >= min_turnos
        and pct_locutores_recorrentes
        >= min_pct_locutores_recorrentes
    )

    return {
        "is_teatro": is_teatro,
        "qtd_candidatos_locutor": len(
            candidatos
        ),
        "qtd_locutores_recorrentes": len(
            locutores_recorrentes
        ),
        "qtd_turnos": qtd_turnos,
        "pct_locutores_recorrentes": (
            pct_locutores_recorrentes
        ),
        "locutores": sorted(
            locutores_recorrentes
        ),
        "contagem_locutores": {
            locutor: contagem[locutor]
            for locutor
            in sorted(
                locutores_recorrentes
            )
        },
    }

In [19]:
resultados_teatro = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        continue

    analise = (
        avaliar_estrutura_teatral(
            txt_path
        )
    )

    resultados_teatro.append({
        "titulo": row["titulo"],
        **analise,
    })


df_teatro = pd.DataFrame(
    resultados_teatro
)

df_teatro[
    df_teatro["is_teatro"]
].sort_values(
    "pct_locutores_recorrentes",
    ascending=False,
)

,titulo,is_teatro,qtd_candidatos_locutor,qtd_locutores_recorrentes,qtd_turnos,pct_locutores_recorrentes,locutores,contagem_locutores
108,Lição de Botânica,True,462,2,461,0.997835,"[barão, d]","{'barão': 117, 'd': 344}"
134,O Anel de Polícrates,True,46,2,45,0.978261,"[a, z]","{'a': 23, 'z': 22}"
100,"Hoje Avental, amanhã Luva",True,298,3,290,0.973154,"[bento, durval, rosinha]","{'bento': 38, 'durval': 121, 'rosinha': 131}"
84,Filosofia de um par de botas,True,69,2,67,0.971014,"[bota direita, bota esquerda]","{'bota direita': 34, 'bota esquerda': 33}"
35,As Forças Caudinas,True,675,6,655,0.970370,"[coronel, emília, margarida, seabra, tito, v]","{'coronel': 64, 'emília': 180, 'margarida': 12..."
66,Desencantos,True,618,5,599,0.969256,"[clara, clara, pedro alves, luís, pedro alves, v]","{'clara': 204, 'clara, pedro alves': 3, 'luís'..."
130,Não consultes Médico,True,355,3,344,0.969014,"[cav, d, mag]","{'cav': 106, 'd': 180, 'mag': 58}"
140,O caminho da porta,True,411,5,397,0.965937,"[carlota, doutor, inocêncio, v, valentim]","{'carlota': 122, 'doutor': 104, 'inocêncio': 4..."
178,Os deuses de casaca,True,314,7,302,0.961783,"[apolo, cupido, júpiter, marte, mercúrio, prot...","{'apolo': 40, 'cupido': 64, 'júpiter': 70, 'ma..."
219,"Tu, só tu, puro amor",True,260,4,250,0.961538,"[cam, camões, d, d man]","{'cam': 59, 'camões': 4, 'd': 183, 'd man': 4}"


Bom com algumas mudanças na forma que os personagens são agrupados, ainda tivemos as correspondencias entrando, mas conforme, transformamos em um script, tiraremos estas 

* Bons dias
* Correspondencia de Machado de Assis

#### Cartas

Importando a heuristicas de cartas que tinhamos no outro notebook de N assinaturas para considerar cartas

In [20]:
def normalizar_texto(
    texto: str,
) -> str:

    texto = unicodedata.normalize(
        "NFKD",
        str(texto),
    )

    texto = "".join(
        c
        for c in texto
        if not unicodedata.combining(c)
    )

    texto = texto.lower()

    texto = re.sub(
        r"[^\w\s]",
        " ",
        texto,
    )

    texto = re.sub(
        r"\s+",
        " ",
        texto,
    )

    return texto.strip()


def normalizar_linha(
    texto: str,
) -> str:

    return normalizar_texto(
        texto
    )

In [21]:
PADROES_ASSINATURA = {
    normalizar_linha(x)
    for x in [
        "MACHADO DE ASSIS",
        "M. DE ASSIS",
        "M DE ASSIS",
        "MACHADO D'ASSIS",
        "J. M. MACHADO DE ASSIS",
        "JOAQUIM MARIA MACHADO DE ASSIS",
    ]
}


def encontrar_assinaturas_machado(
    texto: str,
) -> list[dict]:

    assinaturas = []

    linhas = texto.splitlines()

    for i, linha in enumerate(
        linhas
    ):
        linha_norm = normalizar_linha(
            linha
        )

        if (
            linha_norm
            not in PADROES_ASSINATURA
        ):
            continue

        inicio = max(
            0,
            i - 3,
        )

        fim = min(
            len(linhas),
            i + 2,
        )

        assinaturas.append({
            "assinatura": linha.strip(),
            "contexto": "\n".join(
                linhas[inicio:fim]
            ),
        })

    return assinaturas

In [22]:
def avaliar_carta(
    txt_path,
    qtd_assinaturas_minima=2,
) -> dict:

    txt_path = Path(
        txt_path
    )

    texto = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

    assinaturas = (
        encontrar_assinaturas_machado(
            texto
        )
    )

    return {
        "is_carta": (
            len(assinaturas) > qtd_assinaturas_minima
        ),
        "qtd_assinaturas_machado": len(
            assinaturas
        ),
        "assinaturas": assinaturas,
    }

In [23]:
resultados_cartas = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    resultado = avaliar_carta(
        txt_path
    )

    resultados_cartas.append({
        "titulo": row["titulo"],
        **resultado,
    })


df_cartas = pd.DataFrame(
    resultados_cartas
)

df_cartas[
    df_cartas["is_carta"]
].sort_values(
    "qtd_assinaturas_machado",
    ascending=False,
)

,titulo,is_carta,qtd_assinaturas_machado,assinaturas
57,Correspondência de Machado de Assis,True,23,"[{'assinatura': 'MACHADO DE ASSIS.', 'contexto..."
183,Pareceres de Machado de Assis,True,16,"[{'assinatura': 'Machado de Assis', 'contexto'..."
187,Polêmicas e reflexões,True,4,"[{'assinatura': 'MACHADO DE ASSIS', 'contexto'..."
124,Na Academia Brasileira de Letras,True,3,"[{'assinatura': 'Machado de Assis', 'contexto'..."


Os dois ultimos são respectivamente, coletaneas e discursos de machado, para cartas, vamos usar um minimo de 5 assinaturas, teremos só 2 obras

In [24]:
resultados_cartas = []

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    resultado = avaliar_carta(
        txt_path,
        qtd_assinaturas_minima=5,
    )

    resultados_cartas.append({
        "titulo": row["titulo"],
        **resultado,
    })


df_cartas = pd.DataFrame(
    resultados_cartas
)

df_cartas[
    df_cartas["is_carta"]
].sort_values(
    "qtd_assinaturas_machado",
    ascending=False,
)

,titulo,is_carta,qtd_assinaturas_machado,assinaturas
57,Correspondência de Machado de Assis,True,23,"[{'assinatura': 'MACHADO DE ASSIS.', 'contexto..."
183,Pareceres de Machado de Assis,True,16,"[{'assinatura': 'Machado de Assis', 'contexto'..."


Fechamos legal aqui também

#### Coletaneas 

Importando a heuristica também do que já haviamos feito. 

Recapitulando: 
* encontrar títulos dentro da obra
* verificar se temos N% do titulo original dentro da obra

In [25]:
def encontrar_titulos(
    texto: str,
    titulo_atual: str,
    titulos_normalizados: dict,
) -> list[str]:

    titulo_atual_norm = (
        normalizar_texto(
            titulo_atual
        )
    )

    texto_norm = normalizar_texto(
        texto
    )

    encontrados = {
        titulo_original
        for (
            titulo_original,
            titulo_norm,
        )
        in titulos_normalizados.items()
        if (
            titulo_norm
            != titulo_atual_norm

            and titulo_norm
            in texto_norm
        )
    }

    return sorted(
        encontrados
    )

In [26]:
def encontrar_ancoras_titulo(
    texto: str,
    titulo: str,
) -> list[dict]:

    titulo_norm = normalizar_texto(
        titulo
    )

    linhas = texto.splitlines()

    candidatos = []

    pos_acumulada = 0

    for i, linha in enumerate(
        linhas
    ):
        linha_norm = (
            normalizar_texto(
                linha
            )
        )

        if (
            titulo_norm
            not in linha_norm
        ):
            pos_acumulada += (
                len(linha) + 1
            )
            continue

        titulo_isolado = (
            linha_norm
            == titulo_norm
        )

        linha_seguinte_vazia = (
            i + 1 < len(linhas)
            and not linhas[
                i + 1
            ].strip()
        )

        prioridade = 0

        if titulo_isolado:
            prioridade += 2

        if linha_seguinte_vazia:
            prioridade += 1

        candidatos.append({
            "linha": i,
            "posicao": pos_acumulada,
            "titulo_isolado": (
                titulo_isolado
            ),
            "linha_seguinte_vazia": (
                linha_seguinte_vazia
            ),
            "prioridade": prioridade,
        })

        pos_acumulada += (
            len(linha) + 1
        )

    return sorted(
        candidatos,
        key=lambda x: (
            -x["prioridade"],
            x["posicao"],
        ),
    )

In [27]:
def comparar_fragmento_com_original(
    texto_documento: str,
    titulo_encontrado: str,
    titulos_info: dict,
    tamanho_fragmento: int = 1000,
    tamanho_minimo: int = 100,
) -> dict | None:

    titulo_norm = normalizar_texto(
        titulo_encontrado
    )

    ancoras = encontrar_ancoras_titulo(
        texto_documento,
        titulo_encontrado,
    )

    if not ancoras:
        return None

    info_original = (
        titulos_info.get(
            titulo_norm
        )
    )

    if info_original is None:
        return None

    texto_original = (
        info_original[
            "texto_bronze"
        ]
    )

    texto_original_norm = (
        normalizar_texto(
            texto_original
        )
    )

    pos_original = (
        texto_original_norm.find(
            titulo_norm
        )
    )

    if (
        pos_original != -1
        and pos_original < 500
    ):
        texto_original_conteudo = (
            texto_original_norm[
                pos_original
                + len(titulo_norm):
            ]
            .strip()
        )
    else:
        texto_original_conteudo = (
            texto_original_norm
        )

    if not texto_original_conteudo:
        return None

    resultados_ancoras = []

    linhas_documento = (
        texto_documento.splitlines()
    )

    for ancora in ancoras:

        linhas_depois = (
            linhas_documento[
                ancora["linha"] + 1:
            ]
        )

        texto_depois = "\n".join(
            linhas_depois
        )

        texto_depois_norm = (
            normalizar_texto(
                texto_depois
            )
        )

        tamanho_real = min(
            tamanho_fragmento,
            len(
                texto_original_conteudo
            ),
            len(
                texto_depois_norm
            ),
        )

        if (
            tamanho_real
            < tamanho_minimo
        ):
            continue

        fragmento_documento = (
            texto_depois_norm[
                :tamanho_real
            ]
        )

        fragmento_original = (
            texto_original_conteudo[
                :tamanho_real
            ]
        )

        similaridade = (
            SequenceMatcher(
                None,
                fragmento_documento,
                fragmento_original,
            )
            .ratio()
        )

        resultados_ancoras.append({
            "similaridade": similaridade,
            "tamanho_fragmento_usado": (
                tamanho_real
            ),
            "titulo_isolado": ancora[
                "titulo_isolado"
            ],
            "linha_seguinte_vazia": (
                ancora[
                    "linha_seguinte_vazia"
                ]
            ),
            "prioridade_ancora": ancora[
                "prioridade"
            ],
        })

    if not resultados_ancoras:
        return None

    melhor = max(
        resultados_ancoras,
        key=lambda x: (
            x["prioridade_ancora"],
            x["similaridade"],
        ),
    )

    return {
        "titulo": titulo_encontrado,
        **melhor,
        "qtd_ancoras_encontradas": len(
            ancoras
        ),
    }

In [28]:
def avaliar_coletanea(
    texto: str,
    titulo_atual: str,
    titulos_normalizados: dict,
    titulos_info: dict,
    tamanho_fragmento: int = 1000,
    threshold: float = 0.10,
    qtd_necessaria: int = 2,
) -> dict:

    titulos_encontrados = (
        encontrar_titulos(
            texto=texto,
            titulo_atual=titulo_atual,
            titulos_normalizados=(
                titulos_normalizados
            ),
        )
    )

    resultados = []

    for titulo_encontrado in (
        titulos_encontrados
    ):
        resultado = (
            comparar_fragmento_com_original(
                texto_documento=texto,
                titulo_encontrado=(
                    titulo_encontrado
                ),
                titulos_info=(
                    titulos_info
                ),
                tamanho_fragmento=(
                    tamanho_fragmento
                ),
            )
        )

        if resultado is None:
            continue

        resultados.append({
            "titulo": titulo_encontrado,
            "similaridade": resultado[
                "similaridade"
            ],
        })

    similares = [
        resultado
        for resultado in resultados
        if (
            resultado["similaridade"]
            > threshold
        )
    ]

    titulos_similares = [
        resultado["titulo"]
        for resultado in similares
    ]

    return {
        "is_coletanea": (
            len(titulos_similares)
            >= qtd_necessaria
        ),
        "qtd_titulos_encontrados": len(
            titulos_encontrados
        ),
        "qtd_titulos_similares": len(
            titulos_similares
        ),
        "titulos_similares": (
            titulos_similares
        ),
        "similaridades_titulos": (
            resultados
        ),
    }

In [29]:
textos_bronze = {}

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    textos_bronze[
        row["titulo"]
    ] = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )

In [30]:
titulos_normalizados = {
    titulo: normalizar_texto(
        titulo
    )
    for titulo in (
        df_obras["titulo"]
        .dropna()
        .unique()
    )
}
titulos_info = {
    normalizar_texto(titulo): {
        "titulo": titulo,
        "texto_bronze": texto,
    }
    for titulo, texto
    in textos_bronze.items()
}

In [31]:
resultados_coletaneas = []

for _, row in df_obras.iterrows():

    texto = textos_bronze[
        row["titulo"]
    ]

    resultado = avaliar_coletanea(
        texto=texto,
        titulo_atual=row["titulo"],
        titulos_normalizados=(
            titulos_normalizados
        ),
        titulos_info=titulos_info,
        threshold=0.10,
        qtd_necessaria=2,
    )

    resultados_coletaneas.append({
        "titulo": row["titulo"],
        **resultado,
    })


df_coletaneas = pd.DataFrame(
    resultados_coletaneas
)

In [32]:
df_coletaneas[
    df_coletaneas["is_coletanea"]
].sort_values(
    "qtd_titulos_similares",
    ascending=False,
)

,titulo,is_coletanea,qtd_titulos_encontrados,qtd_titulos_similares,titulos_similares,similaridades_titulos
253,Várias Histórias,True,34,17,"[A Cartomante, A Causa Secreta, A Desejada das...","[{'titulo': 'A Carteira', 'similaridade': 0.01..."
192,Páginas Recolhidas,True,32,11,"[Caso da Vara, Entre 1892 e 1894, Eterno!, Hen...","[{'titulo': 'A Carteira', 'similaridade': 0.02..."
181,Papéis Avulsos,True,35,11,"[A Chinela Turca, A Sereníssima República, D. ...","[{'titulo': 'A Carteira', 'similaridade': 0.02..."
202,Relíquias da Casa Velha,True,30,10,"[Anedota do Cabriolet, Evolução, Marcha Fúnebr...","[{'titulo': 'A Carteira', 'similaridade': 0.02..."
55,Contos Fluminenses,True,29,8,"[A Mulher de Preto, Confissões de uma Viúva, F...","[{'titulo': 'A Carteira', 'similaridade': 0.02..."
98,Histórias da Meia-Noite,True,25,8,"[A Ela, A Parasita Azul, As Bodas de Luís Duar...","[{'titulo': 'A Carteira', 'similaridade': 0.01..."
187,Polêmicas e reflexões,True,22,3,"[Elogio da vaidade, Filosofia de um par de bot...","[{'titulo': 'A Ela', 'similaridade': 0.035}, {..."
129,Novas relíquias,True,23,2,"[Antes da Missa, Filosofia de um par de botas]","[{'titulo': 'A Ela', 'similaridade': 0.031}, {..."
99,Histórias sem Data,True,16,2,"[A Igreja do Diabo, O Lapso]","[{'titulo': 'A Causa Secreta', 'similaridade':..."


Tudo ok por aqui 

#### Final 

Classificamos as obras em 5 tipos de categorias

* Cartas
* Coletaneas
* Autos de teatro
* Romances
* Criticas
    * Esses aqui encontramos pelo nome e verificação manual, não haverão heuristicas pra eles

Hora de verificar a distribuição desses tipos no nosso dataset de fato

In [33]:
CRITICAS_MANUAIS = {
    "textos críticos",
    "críticas teatrais",
}


textos_bronze = {}

for _, row in df_obras.iterrows():

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        continue

    textos_bronze[
        row["titulo"]
    ] = txt_path.read_text(
        encoding="utf-8",
        errors="ignore",
    )


titulos_normalizados = {
    titulo: normalizar_texto(
        titulo
    )
    for titulo in (
        df_obras["titulo"]
        .dropna()
        .unique()
    )
}


titulos_info = {
    normalizar_texto(titulo): {
        "titulo": titulo,
        "texto_bronze": texto,
    }
    for titulo, texto
    in textos_bronze.items()
}


In [34]:
resultados_classificacao = []

for _, row in df_obras.iterrows():

    titulo = row["titulo"]

    txt_path = (
        PROJECT_ROOT
        / row["bronze_text_path"]
    )

    if not txt_path.exists():
        continue

    texto = textos_bronze[
        titulo
    ]


    is_critica = (
        titulo.strip().casefold()
        in CRITICAS_MANUAIS
    )
    
    resultado_carta = avaliar_carta(
        txt_path,
        qtd_assinaturas_minima=5,
    )

    resultado_coletanea = avaliar_coletanea(
        texto=texto,
        titulo_atual=titulo,
        titulos_normalizados=(
            titulos_normalizados
        ),
        titulos_info=(
            titulos_info
        ),
        threshold=0.10,
        qtd_necessaria=2,
    )

   
    resultado_teatro = (
        avaliar_estrutura_teatral(
            txt_path,
            min_pct_locutores_recorrentes=0.91,
        )
    )

    

    resultado_poema = (
        avaliar_estrutura_poetica(
            txt_path,
            max_palavras_linha=10,
            min_pct_linhas_curtas=0.85,
        )
    )

    # A prioridade é necessária porque as
    # heurísticas não são mutuamente exclusivas.

    if is_critica:
        tipo_obra = "Crítica"

    elif resultado_carta["is_carta"]:
        tipo_obra = "Carta"

    elif resultado_coletanea["is_coletanea"]:
        tipo_obra = "Coletânea"

    elif resultado_teatro["is_teatro"]:
        tipo_obra = "Teatro"

    elif resultado_poema["is_poema"]:
        tipo_obra = "Poema/Cântico"

    else:
        tipo_obra = "Romance/Conto"


    resultados_classificacao.append({
        "titulo": titulo,
        "tipo_obra": tipo_obra,

        "is_critica": is_critica,
        "is_carta": resultado_carta[
            "is_carta"
        ],
        "is_coletanea": resultado_coletanea[
            "is_coletanea"
        ],
        "is_teatro": resultado_teatro[
            "is_teatro"
        ],
        "is_poema": resultado_poema[
            "is_poema"
        ],

        "qtd_assinaturas_machado": (
            resultado_carta[
                "qtd_assinaturas_machado"
            ]
        ),

        "qtd_titulos_similares": (
            resultado_coletanea[
                "qtd_titulos_similares"
            ]
        ),

        "pct_locutores_recorrentes": (
            resultado_teatro[
                "pct_locutores_recorrentes"
            ]
        ),

        "pct_linhas_curtas": (
            resultado_poema[
                "pct_linhas_curtas"
            ]
        ),
    })


df_classificacao = pd.DataFrame(
    resultados_classificacao
)

In [35]:
distribuicao = (
    df_classificacao[
        "tipo_obra"
    ]
    .value_counts()
    .rename_axis(
        "tipo_obra"
    )
    .reset_index(
        name="quantidade"
    )
)

distribuicao["percentual"] = (
    distribuicao["quantidade"]
    / len(df_classificacao)
    * 100
).round(2)


print(
    f"Total de obras classificadas: "
    f"{len(df_classificacao)}"
)

display(
    distribuicao
)

Total de obras classificadas: 257


,tipo_obra,quantidade,percentual
0,Romance/Conto,220,85.60
1,Teatro,16,6.23
2,Coletânea,9,3.50
3,Poema/Cântico,8,3.11
4,Carta,2,0.78
5,Crítica,2,0.78


#### Conclusão 

Conseguimos classificar bem em 5 tipos de obra textual que machado costuma escrever, embora não tenhamos heuristíca para as critícas, temos o conhecimento prévio de duas e para nossa aplicação isso deve bastar.

Descobrimos ao longo do processo o tipo de texto teatro e até que temos bastante 16 peças contra 8 poemas que era o que estavamos caçando inicialmente. Certamente será uma classificação bem vinda ao processo!